In [86]:
from pypdf import PdfReader
import re
from pprint import pprint
from sentence_transformers import SentenceTransformer
import os 
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [87]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [88]:
def clean_text(full_text):
    #removes boilerplate 
    full_text = re.sub(r"This content.*?/terms", "", full_text, flags=re.DOTALL)
    full_text = re.sub(r"THE MATHE.*?ERICA", "", full_text, flags=re.DOTALL)
    full_text = re.sub(r"JUGGLING.*?2005", "", full_text, flags=re.DOTALL)
    text_list = []
    for line in full_text.split("\n"):
        if "\x00" in line: #removes nullspace
            line = line.replace("\x00",'')
            text_list.append(line)
        elif not line.strip():  # empty or only whitespace
            continue
        else: #adds items back to list
            text_list.append(line)
    cleaned_text = "\n".join(text_list)
    return cleaned_text

In [89]:
def pdf_to_text(file_name):
    '''Takes in file name and processes it as a text string'''
    full_text = ''
    with open(file_name,"rb") as file:
        reader = PdfReader(file)
        # Loop through all pages
        for page in reader.pages: #page is an object of pageobject class
            page_text = page.extract_text()
            full_text+=" \n" + page_text
    return full_text


In [90]:
file_name = "Warrington-JugglingProbabilities-2005.pdf"
text = pdf_to_text(file_name)
cleaned_text = clean_text(text)

In [91]:
# pprint(cleaned_text)
def chunk_text(text, chunk_size, overlap):
    text_list = []
    i = 0
    while i < len(text):
        # print(f'Start {i}, End {i + chunk_size}')
        text_list.append( text[i:i+chunk_size])
        i = i + (chunk_size - overlap)
    return text_list
        

In [92]:
#Loading a pretrained Sentence Transformer model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
def embed_chunks(chunks,model):
    # Calculate embeddings by calling model.encode()
    embeddings = model.encode(chunks)
    print(embeddings.shape)
    
    chunk_embs = [{"text": chunk, "embeddings": embedding} for chunk,embedding in zip(chunks,embeddings)]
    return chunk_embs
chunk_embs = embed_chunks(chunk_text(cleaned_text, 800, 100),model)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6221.30it/s]


(50, 384)


In [93]:
#shape of first embedding
print(chunk_embs[0]['embeddings'].shape)
print(chunk_embs[0]['embeddings'][:5])
for item in chunk_embs:
    if 'Markov chain' in item['text']:
        numbers = item['embeddings'][:5]
        # print(numbers)
        break


(384,)
[-0.0702213  -0.0676851  -0.02677578  0.00544441  0.03039066]


In [94]:
import numpy as np
def search(question, chunk_embs, model, k=5):
    '''returns top k answers to question based on similarity score'''
    q_embedding = model.encode(question)
    embeddings = [item['embeddings'] for item in chunk_embs]
    sim = model.similarity(q_embedding,embeddings) #gives similarity scores
    indices = np.argsort(-sim) #finding the arrays with highest similarity values
    
    return [{'text':chunk_embs[i]['text'], 'score': sim[0,i].item()} for i in indices[0,:k]]

search('what is a Markov chain?', chunk_embs, model)

[{'text': 'When MC(Sth,f, P) is in state i, we define our new process to be in the state \n [rh (~)]. It follows that its states are {[v]}sth, . This is a lumped process derived from \n MC(Sth,f, P) by grouping together certain states (see Figure 4). \n Figure 4. The Markov chain MC(St3,1, P) along with the corresponding lumped process. \n Certainly Magnus wanders randomly among the states of this new process. We do \n not yet know, however, that this new process is a Markov chain. In particular, it is not \n clear that the probability of transitioning from [v] to [w] is independent of how long \n Magnus has been in [v]. If it were dependent, then the transition probabilities would \n ]  113 \n depend on previous states as well as the current state. Or, stated another way, the \n transition probability from [v',
  'score': 0.5882135629653931},
 {'text': 'obabilities depend only on the current state. To de- \n scribe a Markov chain, we need to know the possible states Q and the possible

In [95]:
#testing the results
for q in [
    "What is a Markov chain?",
    "Who is the author?",
    "How does juggling work?",
    "Banana sandwich recipe",
]:
    print(f"\n=== {q} ===")
    results = search(q, chunk_embs, model, k=3)
    for r in results:
        print(f"  ----{r['score']:.3f}--- | {r['text'][:100]}")


=== What is a Markov chain? ===
  ----0.588--- | When MC(Sth,f, P) is in state i, we define our new process to be in the state 
 [rh (~)]. It follows
  ----0.566--- | obabilities depend only on the current state. To de- 
 scribe a Markov chain, we need to know the po
  ----0.493--- | atter Markov chain. The 
 first step is to find the vector /3 of steady-state probabilities for MC(S

=== Who is the author? ===
  ----0.224--- | Juggling Probabilities 
Author(s): Gregory S. Warrington 
Source: The American Mathematical Monthly 
  ----0.168--- |  He is reputed to be the first person to 
 juggle five clubs. Reliable details of his life are outcl
  ----0.154--- | 7
 Figure 2. The state graph G5,2.
 Let us revisit the scenario introduced at the beginning of the p

=== How does juggling work? ===
  ----0.658--- | bility of making each legal throw. 
 There are countless ways to make these specifications, but we b
  ----0.576--- |  leg or behind the back when making a catch will not be noted. 

In [96]:
def build_prompt(question, retrieved_chunks):
    '''Pulls out text from retrieved chunks and labels final string as question and context'''
    text_list = [item['text'] for item in retrieved_chunks]
    return 'Context: ' + '---\n---'.join(text_list) + '---\n---' 'Question: ' + question

In [97]:
results = search(q, chunk_embs, model, k=8)
q = "What is a Markov chain?"
c = build_prompt(q, results)

In [ ]:
import time

def ask_llm(question, retrieved_chunks, client, max_retries=3):
    prompt = build_prompt(question, retrieved_chunks)
    
    for attempt in range(max_retries):
        time.sleep(20)
        try:
            response = client.models.generate_content(
                model="gemini-3-flash-preview",
                config=types.GenerateContentConfig(
                    system_instruction="You are a helpful assistant. Use only the provided context to answer the user's question. If the context does not contain the answer, say so honestly."
                ),
                contents=prompt,
            )
            return response.text
        except Exception as e:
            if attempt == max_retries - 1:
                raise  # last attempt — let it crash
            wait_time = 5 ** attempt  # 1s, 25s, 125s — exponential backoff a^x
            print(f"  API error: {e}. Retrying in {wait_time}s...")
            time.sleep(wait_time)

In [99]:
client = genai.Client()  # once, outside the loop

while True:
    q = input("\nAsk a question (or 'quit'): ").strip()
    if q.lower() == "quit":
        break
    if not q:
        continue
    results = search(q, chunk_embs, model, k=3)
    answer = ask_llm(q, results,client)
    print("\n" + answer)

In [100]:
test_queries = [
    {
        "question": "Who is the author of this paper?",
        "category": "easy_factual",
        "expected_keywords": ["Warrington","Gregory"],   # answer must include at least one of these
        "should_answer": True,                  # the doc contains this info
    },
    {
        "question": "What year was the paper published?",
        "category": "easy_factual",
        "expected_keywords": ["2005", "February"],   # answer must include at least one of these
        "should_answer": True,                  # the doc contains this info
    },
    {
        "question": "What are the references",
        "category": "metadata",
        "expected_keywords": ["Buhler","Kemeny","jugglingdb"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "How many pages are there?",
        "category": "metadata",
        "expected_keywords": ["14","15"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What is a markov chain",
        "category": "definition",
        "expected_keywords": ['discrete-time',"random","transition", "probabilities"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What is the juggling process",
        "category": "definition",
        "expected_keywords": ["throw","ball","hand"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What is theorem 1, explain it?",
        "category": "detail",
        "expected_keywords": ["fraction", "A(v)"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What are the differences between the standard model and the add-drop model?",
        "category": "synthesis",
        "expected_keywords": ["generalizes","drop", "vertex"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "What's the main idea?",
        "category": "tricky_semantic",
        "expected_keywords": ["random","fraction","time"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "Who first considered the juggling state graph?",
        "category": "specific_question",
        "expected_keywords": ["Boyce"],
        "should_answer": True,                 
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "Banana sandwich recipe",
        "category": "negative",
        "expected_keywords": [],
        "should_answer": False,                 # model should refuse
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    {
        "question": "Did the egg come before the chicken?",
        "category": "negative",
        "expected_keywords": [],
        "should_answer": False,                 # model should refuse
        "refusal_keywords": ["does not", "doesn't", "no information", "not provided", "not in the context"],
    },
    
]

In [101]:
def score_query(test_query, answer):
    answer_lower = answer.lower()
    
    if test_query["should_answer"]:
        # Positive test: look for any expected keyword in the answer
        matches = [kw for kw in test_query["expected_keywords"] 
                   if kw.lower() in answer_lower]
        passed = len(matches) > 0
        if passed:
            reason = f"found keywords: {matches}"
        else:
            reason = f"missing all expected keywords: {test_query['expected_keywords']}"
    else:
        # Negative test: look for any refusal phrase in the answer
        refusals = [kw for kw in test_query["refusal_keywords"] 
                    if kw.lower() in answer_lower]
        passed = len(refusals) > 0
        if passed:
            reason = f"refused with: {refusals}"
        else:
            reason = "did not refuse"
    
    return {
        "question": test_query["question"],
        "category": test_query["category"],
        "answer": answer,
        "passed": passed,
        "reason": reason,
    }

In [102]:
# 1. Positive test, model answered correctly
print(score_query(test_queries[0], "The author is Gregory S. Warrington."))

# 2. Positive test, model answered wrong
print(score_query(test_queries[0], "I don't know."))

# 3. Negative test, model refused
neg_test = next(t for t in test_queries if not t["should_answer"])
print(score_query(neg_test, "The document does not contain information about this."))

# 4. Negative test, model failed to refuse
print(score_query(neg_test, "Here is a banana sandwich recipe."))

{'question': 'Who is the author of this paper?', 'category': 'easy_factual', 'answer': 'The author is Gregory S. Warrington.', 'passed': True, 'reason': "found keywords: ['Warrington', 'Gregory']"}
{'question': 'Who is the author of this paper?', 'category': 'easy_factual', 'answer': "I don't know.", 'passed': False, 'reason': "missing all expected keywords: ['Warrington', 'Gregory']"}
{'question': 'Banana sandwich recipe', 'category': 'negative', 'answer': 'The document does not contain information about this.', 'passed': True, 'reason': "refused with: ['does not']"}
{'question': 'Banana sandwich recipe', 'category': 'negative', 'answer': 'Here is a banana sandwich recipe.', 'passed': False, 'reason': 'did not refuse'}


In [103]:
def run_eval(test_queries, chunk_embs, model, client):
    """Run all queries through the pipeline. Return list of result dicts."""
    # For each query: call search → ask_llm → score_query → collect
    eval_results = []
    for query in test_queries:
    
        question = query['question']
        # category = query['category']
        # ex_keywords = query['expected_keywords']
        # sh_answer = query['should_answer']
        # ref_keywords = query['refusal_keywords']

        results = search(question, chunk_embs, model, k=3)
        answer = ask_llm(question, results,client)
        score_result = score_query(query,answer)
        eval_results.append(score_result)
        print(f"[{len(eval_results)}/{len(test_queries)}] {'Correct' if score_result['passed'] else 'Incorrect'} {question}")
    return eval_results
    

In [105]:
run_eval(test_queries, chunk_embs, model, client)

[1/12] Correct Who is the author of this paper?
[2/12] Correct What year was the paper published?
[3/12] Incorrect What are the references
[4/12] Correct How many pages are there?
[5/12] Correct What is a markov chain
[6/12] Correct What is the juggling process
  API error: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash\nPlease retry in 42.000658917s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3-flash\nPlease retry in 35.415806765s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-3-flash', 'location': 'global'}, '

In [ ]:
def print_summary(results):
    """Pretty-print the results: per-query + overall + per-category."""
    for result in results:
        pprint(result)